# 🧬 SPS Self-Specialization — Dynamic Serialization + Capability Handler Demo

**Research claim demonstrated:** the system starts with only `IntegerMultiplication`; when `FloatMultiplication` is missing, it dynamically serializes/generalizes the existing capability, creates `SerializeCapability [S0]`, reparents the existing integer capability, creates a transient `S0-C` copy, specializes the copy with an external AI model, verifies it, activates it as `S1`, persists it, reloads it, and reuses it.

This notebook follows the current `main` branch implementation and makes the runtime creation of the general capability explicit.

## 🔬 Complete research flow

```text
INITIAL
IntegerMultiplication [S1]
        │
        │ request multiply(2.5, 4.0)
        ▼
FloatMultiplication missing
        │
        ▼
SERIALIZE / GENERALIZE existing IntegerMultiplication
        │
        ├── create SerializeCapability [S0]
        └── reparent existing IntegerMultiplication
        │
        ▼
REPLICATE → transient copy [S0-C]
        ↓
SPECIALIZE → Ollama + Qwen Coder
        ↓
GENERATED → FloatMultiplication
        ↓
VERIFY → syntax/policy + functional cases
        ↓
ACTIVATE → FloatMultiplication [S1]
        ↓
FINAL
              SerializeCapability [S0]
                        │
               ┌────────┴────────┐
               ▼                 ▼
 IntegerMultiplication    FloatMultiplication
        [S1]                    [S1]
        
PERSIST → RELOAD → REUSE without another AI call
```

**Important:** `SerializeCapability` does not exist before the float request.

## 1. Install and start Ollama first

**Run this cell first.** Ollama is started from `/content`, a stable directory that is not deleted when the repository is recloned. This avoids the previous `llama-server process has terminated / cannot get current path` failure.

In [ ]:
%cd /content
!apt-get update -qq
!apt-get install -y -qq zstd curl
!curl -fsSL https://ollama.com/install.sh | sh
!pkill -9 ollama || true
!pkill -9 llama-server || true
!nohup ollama serve >/tmp/ollama.log 2>&1 &
!sleep 5
!ollama --version
!curl -sf http://127.0.0.1:11434/api/tags || (cat /tmp/ollama.log; exit 1)

# Pull the local coding model used by the prototype.
!ollama pull qwen2.5-coder:7b

## 2. Clone the latest `main` branch and install dependencies

The notebook deliberately performs a fresh clone so an older Colab checkout cannot shadow the implementation.

In [ ]:
%cd /content
!rm -rf self-specialization
!git clone --branch main --single-branch -q https://github.com/muhammadnaumantahir/self-specialization.git
%cd /content/self-specialization
!pip -q install -r requirements.txt pytest
!echo 'Repository commit:'
!git rev-parse HEAD
!echo '\nAvailable Ollama models:'
!ollama list

## 3. Verify the implementation before running the real model

The deterministic test suite validates dynamic serialization, reparenting, `S0-C` replication, specialization, verification, the final sibling hierarchy, persistence, reload, and failure diagnostics. It does not require Ollama.

In [ ]:
%cd /content/self-specialization
!PYTHONPATH=. pytest -q

## 4. Run the real SPS experiment

The demo creates a fresh persistent registry under `/tmp/sps-capability-registry` for every run. The initial registry therefore contains only `IntegerMultiplication [S1]`.

In [ ]:
%cd /content/self-specialization
import os
os.environ['OLLAMA_MODEL'] = 'qwen2.5-coder:7b'
!PYTHONPATH=. python experiments/self_specialization_demo.py

## 5. What the demo must prove

### Initial capability hierarchy

Before the float request, the persistent hierarchy must be exactly:

```text
IntegerMultiplication [S1]
```

There must be **no** `SerializeCapability` and no `FloatMultiplication` yet.

### Runtime serialization/generalization

When `[float, float] → float` is missing:

1. Existing `IntegerMultiplication` is selected as the source.
2. `SerializeCapability [S0]` is created dynamically.
3. The existing integer capability keeps its ID and becomes a child of the new S0 capability.
4. A transient `S0-C` copy is created for specialization.
5. Qwen generates `FloatMultiplication`.
6. Verification gates activation.
7. The generated capability becomes `S1` and is linked as a sibling of integer multiplication.

### Final capability hierarchy

```text
              SerializeCapability [S0]
                        │
               ┌────────┴────────┐
               ▼                 ▼
 IntegerMultiplication    FloatMultiplication
        [S1]                    [S1]
```

The `S0-C` copy is an internal evolution artifact and must not become a permanent hierarchy node.

### Capability Handler

Each capability owns a lightweight runtime handler containing:

- capability identity
- executable function
- runtime status
- source code resource
- input contract
- output contract

The handler is the prototype's answer to: **when a capability is created, where is its runtime handling and what resources does it own?**

### State transition

```text
IntegerMultiplication [S1]
        │
        │ serialize/generalize
        ▼
SerializeCapability [S0]
        │
        │ replicate source
        ▼
S0-C
        │
        │ specialize
        ▼
GENERATED
        │
        │ verify
        ▼
FloatMultiplication [S1]
```

## 6. Supervisor checklist

Look for these sections in the demo output:

- 🔵 **INITIAL STATE** — only `IntegerMultiplication [S1]` exists.
- 🟡 **NEW REQUEST** — `[float, float] → float` is missing.
- 🧩 **SERIALIZATION / GENERALIZATION** — `SerializeCapability [S0]` is created at runtime.
- 🔗 **REPARENT** — the original integer capability keeps its ID and becomes a child of SerializeCapability.
- 🔁 **REPLICATION** — `IntegerMultiplication` produces a transient `S0-C` copy.
- 🧠 **SPECIALIZATION** — Ollama + `qwen2.5-coder:7b` transforms the copy into the float implementation.
- 🛡️ **VERIFICATION** — generated source is checked before activation.
- 🟢 **STATE 1** — `FloatMultiplication` becomes active.
- 🧩 **HANDLER** — capability identity, status, resources and execution information are visible.
- 🌳 **HIERARCHY** — SerializeCapability has two specialized children: integer and float multiplication.
- 📜 **EVENT TRACE** — serialization, reparenting, replication, specialization, generation, verification, activation and child-link events are visible.
- 💾 **PERSISTENCE** — JSON metadata and Python source are written to the registry.
- 🔄 **RELOAD + REUSE** — the S1 float capability is reconstructed and reused without another AI generation step.

## 7. Persistent research artifacts

```text
/tmp/sps-capability-registry/
├── registry.json                 ← registry index
├── records/<capability-id>.json ← capability metadata/events
└── sources/<id>_<name>.py       ← exact executable capability source
```

The important research result is not simply that Qwen writes Python. The observable sequence is:

**initial capability → capability gap detection → dynamic serialization/generalization → reparent → replication → specialization → verification → activation → hierarchy registration → persistence → reload → reuse**.

## 8. If Ollama fails

Run the diagnostic cell below. It checks the working directory, Ollama processes, API response, and server log.

In [ ]:
%cd /content
!pwd
!echo '\nOllama processes:'
!ps aux | grep -E 'ollama|llama-server' | grep -v grep || true
!echo '\nOllama API:'
!curl -s http://127.0.0.1:11434/api/tags || true
!echo '\nOllama log:'
!cat /tmp/ollama.log